# Module 6 - Alzheimer's: Outcome Analysis

Outcome-code distribution and per-class serious-outcome rates. Serious = DE, LT, HO, DS, CA.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\FDA_FAERS\database\faers.db"
conn = sqlite3.connect(db_path)

## Step 1 - Outcome code distribution across the cohort

In [ ]:
outcomes = pd.read_sql_query("""
    SELECT o.outc_cod,
           COUNT(DISTINCT o.primaryid) AS reports
    FROM outc o
    JOIN (SELECT DISTINCT primaryid FROM alzheimers_analysis) a
        ON o.primaryid = a.primaryid
    GROUP BY o.outc_cod
    ORDER BY reports DESC
""", conn)
outcomes

## Step 2 - Serious outcome rate per class

In [ ]:
# Serious = any of DE, LT, HO, DS, CA on the report.
# LEFT JOIN so reports with NO outcome row still contribute to the denominator.

serious = pd.read_sql_query("""
    SELECT a.drug_class,
           COUNT(DISTINCT a.primaryid) AS total_reports,
           COUNT(DISTINCT CASE
                WHEN o.outc_cod IN ('DE','LT','HO','DS','CA')
                THEN a.primaryid END) AS serious_reports
    FROM alzheimers_analysis a
    LEFT JOIN outc o ON o.primaryid = a.primaryid
    GROUP BY a.drug_class
""", conn)

serious['serious_pct'] = (serious['serious_reports'] / serious['total_reports'] * 100).round(1)
serious

## Step 3 - Grouped bar: outcome codes by class

In [ ]:
outc_by_class = pd.read_sql_query("""
    SELECT a.drug_class, o.outc_cod,
           COUNT(DISTINCT a.primaryid) AS reports
    FROM alzheimers_analysis a
    JOIN outc o ON o.primaryid = a.primaryid
    WHERE o.outc_cod IN ('DE','LT','HO','DS','RI','CA','OT')
    GROUP BY a.drug_class, o.outc_cod
""", conn)

pivot = outc_by_class.pivot(index='outc_cod', columns='drug_class', values='reports').fillna(0)
pivot = pivot.reindex(['DE','LT','HO','DS','RI','CA','OT'])

pivot.plot(kind='bar', figsize=(10, 5))
plt.title('Outcome code distribution by Alzheimer\'s drug class')
plt.xlabel('Outcome code')
plt.ylabel('Distinct reports')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

pivot